# La Liga 2015/16 | Tactical & Performance Analysis
### StatsBomb Open Data | Barcelona vs Real Madrid
**Author:** Adnan Mustafa | Data Analyst

In [ ]:
# Install required libraries
!pip install statsbombpy mplsoccer matplotlib seaborn pandas numpy -q

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import requests
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from mplsoccer import Pitch, VerticalPitch
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")

## 📥 Data Loading & Pipeline

In [ ]:
# Base URL for StatsBomb Open Data
BASE_URL = "https://raw.githubusercontent.com/statsbomb/open-data/master/data/"

In [ ]:
# Step 4: Load competitions & Identify La Liga 2015/16
comp_url = f"{BASE_URL}competitions.json"
df_competitions = pd.read_json(comp_url)
df_competitions['competition_name'] = df_competitions['competition_name'].str.strip()
df_competitions['season_name'] = df_competitions['season_name'].str.strip()

target_season_name = "2015/2016"
target_league = df_competitions[
    (df_competitions['competition_name'] == "La Liga") &
    (df_competitions['season_name'] == target_season_name)
]

comp_id = target_league['competition_id'].iloc[0]
season_id = target_league['season_id'].iloc[0]

print(f"✅ Competition: La Liga | Season: {target_season_name}")
print(f"✅ Competition ID: {comp_id} | Season ID: {season_id}")

In [ ]:
# Fetch metadata for all matches in the identified season
matches_url = f"{BASE_URL}matches/{comp_id}/{season_id}.json"
matches_data = requests.get(matches_url).json()
df_matches = pd.DataFrame(matches_data)

print(f"✅ Total matches loaded for La Liga {target_season_name}: {len(df_matches)}")
# Display first few rows to confirm data structure
df_matches[['match_id', 'match_date', 'home_team', 'away_team']].head()

In [ ]:
# Function to flatten and clean nested event data
def get_cleaned_events(match_id):
    url = f"{BASE_URL}events/{match_id}.json"
    response = requests.get(url)
    events_json = response.json()

    # Use json_normalize to flatten nested columns (like location, pass, shot)
    df = pd.json_normalize(events_json, sep='_')
    return df

# Test with the first match in the list
test_match_id = df_matches['match_id'].iloc[0]
df_test = get_cleaned_events(test_match_id)

print(f"✅ Successfully flattened {len(df_test)} events for Match ID: {test_match_id}")

In [ ]:
# List of Team IDs for Barcelona (217) and Real Madrid (220)
target_teams = ['Barcelona', 'Real Madrid']

# Filter match list for target teams
target_matches = df_matches[
    (df_matches['home_team'].str['home_team_name'].isin(target_teams)) |
    (df_matches['away_team'].str['away_team_name'].isin(target_teams))
]

print(f"✅ Found {len(target_matches)} matches featuring Barcelona or Real Madrid.")

# Professional Batch Processing Loop
all_events_list = []

print("🔄 Starting batch processing of event data...")
for i, match_id in enumerate(target_matches['match_id']):
    # Use our previously defined function
    df_temp = get_cleaned_events(match_id)
    all_events_list.append(df_temp)

    # Progress indicator every 10 matches
    if (i + 1) % 10 == 0:
        print(f"Status: Processed {i + 1} matches...")

# Combine all into one massive Master DataFrame
df_master_events = pd.concat(all_events_list, ignore_index=True)

print(f"✅ Pipeline Complete! Total Events Captured: {len(df_master_events)}")

In [ ]:
# Clean and extract key columns
df_shots = df_master_events[df_master_events['type_name'] == 'Shot'].copy()
df_passes = df_master_events[df_master_events['type_name'] == 'Pass'].copy()

print(f"✅ Total Shots: {len(df_shots)}")
print(f"✅ Total Passes: {len(df_passes)}")
print(f"\nShot columns with data:")
print(df_shots[['player_name','team_name','shot_statsbomb_xg',
                'shot_outcome_name','shot_technique_name',
                'shot_body_part_name','location']].head(5))

## ⚽ Shooting Analysis | Expected Goals (xG)

In [ ]:
# xG Analysis - Top Players by Expected Goals
xg_summary = df_shots.groupby('player_name').agg(
    Total_Shots=('shot_statsbomb_xg', 'count'),
    Total_xG=('shot_statsbomb_xg', 'sum'),
    Goals=('shot_outcome_name', lambda x: (x == 'Goal').sum()),
    Shots_on_Target=('shot_outcome_name', lambda x: x.isin(['Goal','Saved']).sum())
).reset_index()

xg_summary['xG_per_Shot'] = xg_summary['Total_xG'] / xg_summary['Total_Shots']
xg_summary['Goals_vs_xG'] = xg_summary['Goals'] - xg_summary['Total_xG']
xg_summary = xg_summary[xg_summary['Total_Shots'] >= 10].sort_values('Total_xG', ascending=False)

print("✅ Top 15 Players by xG:")
print(xg_summary.head(15).to_string(index=False))

In [ ]:
# xG Bar Chart - Top 15 Players
top15 = xg_summary.head(15).sort_values('Total_xG')

fig, ax = plt.subplots(figsize=(12, 8))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')

bars = ax.barh(top15['player_name'], top15['Total_xG'], color='#00d4ff', alpha=0.85)
ax.barh(top15['player_name'], top15['Goals'], color='#ff4757', alpha=0.7, label='Actual Goals')

ax.set_xlabel('Goals / xG', color='white', fontsize=12)
ax.set_title('Expected Goals (xG) vs Actual Goals\nLa Liga 2015/16 — Barcelona & Real Madrid',
             color='white', fontsize=14, fontweight='bold')
ax.tick_params(colors='white')
ax.xaxis.label.set_color('white')
for spine in ax.spines.values():
    spine.set_edgecolor('#333333')

legend = ax.legend(facecolor='#1a1a2e', labelcolor='white')
blue_patch = mpatches.Patch(color='#00d4ff', alpha=0.85, label='Total xG')
red_patch = mpatches.Patch(color='#ff4757', alpha=0.7, label='Actual Goals')
ax.legend(handles=[blue_patch, red_patch], facecolor='#1a1a2e', labelcolor='white')

plt.tight_layout()
plt.savefig('xg_top_players.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("✅ Chart saved!")

In [ ]:
# Shot Map for Cristiano Ronaldo
cr7_shots = df_shots[df_shots['player_name'].str.contains('Cristiano', na=False)].copy()
cr7_shots['x'] = cr7_shots['location'].apply(lambda loc: loc[0] if isinstance(loc, list) else None)
cr7_shots['y'] = cr7_shots['location'].apply(lambda loc: loc[1] if isinstance(loc, list) else None)
cr7_shots.dropna(subset=['x','y'], inplace=True)

pitch = Pitch(pitch_type='statsbomb', pitch_color='#0d1117', line_color='#ffffff')
fig, ax = pitch.draw(figsize=(12, 8))
fig.patch.set_facecolor('#0d1117')

goals = cr7_shots[cr7_shots['shot_outcome_name'] == 'Goal']
non_goals = cr7_shots[cr7_shots['shot_outcome_name'] != 'Goal']

pitch.scatter(non_goals['x'], non_goals['y'], ax=ax,
              s=non_goals['shot_statsbomb_xg']*800 + 50,
              color='#00d4ff', alpha=0.6, zorder=3, label='No Goal')
pitch.scatter(goals['x'], goals['y'], ax=ax,
              s=goals['shot_statsbomb_xg']*800 + 50,
              color='#ff4757', alpha=0.9, zorder=4, label='Goal')

ax.set_title('Cristiano Ronaldo — Shot Map\nLa Liga 2015/16',
             color='white', fontsize=14, fontweight='bold', pad=15)
ax.legend(facecolor='#1a1a2e', labelcolor='white', loc='upper left')

plt.tight_layout()
plt.savefig('cr7_shot_map.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f"✅ CR7 Shot Map done! Goals: {len(goals)}, Total Shots: {len(cr7_shots)}")

In [ ]:
# Shot Map for Lionel Messi
messi_shots = df_shots[df_shots['player_name'].str.contains('Messi', na=False)].copy()
messi_shots['x'] = messi_shots['location'].apply(lambda loc: loc[0] if isinstance(loc, list) else None)
messi_shots['y'] = messi_shots['location'].apply(lambda loc: loc[1] if isinstance(loc, list) else None)
messi_shots.dropna(subset=['x','y'], inplace=True)

pitch = Pitch(pitch_type='statsbomb', pitch_color='#0d1117', line_color='#ffffff')
fig, ax = pitch.draw(figsize=(12, 8))
fig.patch.set_facecolor('#0d1117')

goals = messi_shots[messi_shots['shot_outcome_name'] == 'Goal']
non_goals = messi_shots[messi_shots['shot_outcome_name'] != 'Goal']

pitch.scatter(non_goals['x'], non_goals['y'], ax=ax,
              s=non_goals['shot_statsbomb_xg']*800 + 50,
              color='#00d4ff', alpha=0.6, zorder=3, label='No Goal')
pitch.scatter(goals['x'], goals['y'], ax=ax,
              s=goals['shot_statsbomb_xg']*800 + 50,
              color='#ff4757', alpha=0.9, zorder=4, label='Goal')

ax.set_title('Lionel Messi — Shot Map\nLa Liga 2015/16',
             color='white', fontsize=14, fontweight='bold', pad=15)
ax.legend(facecolor='#1a1a2e', labelcolor='white', loc='upper left')

plt.tight_layout()
plt.savefig('messi_shot_map.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f"✅ Messi Shot Map done! Goals: {len(goals)}, Total Shots: {len(messi_shots)}")

## 🎯 Passing Analysis

In [ ]:
# Pass Completion Rate
pass_summary = df_passes.groupby(['team_name', 'player_name']).agg(
    Total_Passes=('pass_length', 'count'),
    Incomplete=('pass_outcome_name', lambda x: x.notna().sum())
).reset_index()

pass_summary['Completed'] = pass_summary['Total_Passes'] - pass_summary['Incomplete']
pass_summary['Completion_Rate'] = (
    pass_summary['Completed'] / pass_summary['Total_Passes'] * 100
).round(1)

pass_summary = pass_summary[pass_summary['Total_Passes'] >= 100].sort_values(
    'Completion_Rate', ascending=False)

print("✅ Top 15 Players by Pass Completion Rate:")
print(pass_summary.head(15)[['player_name','team_name',
      'Total_Passes','Completed','Completion_Rate']].to_string(index=False))

In [ ]:
# Pass Completion Bar Chart
top_passers = pass_summary.head(15).sort_values('Completion_Rate')

fig, ax = plt.subplots(figsize=(12, 8))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')

colors = ['#ff4757' if team == 'Real Madrid' else '#00d4ff'
          for team in top_passers['team_name']]

bars = ax.barh(top_passers['player_name'], top_passers['Completion_Rate'],
               color=colors, alpha=0.85)

for bar, val in zip(bars, top_passers['Completion_Rate']):
    ax.text(bar.get_width() - 2, bar.get_y() + bar.get_height()/2,
            f'{val}%', va='center', ha='right', color='white', fontsize=9)

ax.set_xlabel('Pass Completion Rate (%)', color='white', fontsize=12)
ax.set_title('Pass Completion Rate — Top Players\nLa Liga 2015/16 | Barcelona & Real Madrid',
             color='white', fontsize=14, fontweight='bold')
ax.tick_params(colors='white')
ax.set_xlim(0, 105)
for spine in ax.spines.values():
    spine.set_edgecolor('#333333')

blue_patch = mpatches.Patch(color='#00d4ff', alpha=0.85, label='Barcelona')
red_patch = mpatches.Patch(color='#ff4757', alpha=0.85, label='Real Madrid')
ax.legend(handles=[blue_patch, red_patch], facecolor='#1a1a2e', labelcolor='white')

plt.tight_layout()
plt.savefig('pass_completion.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("✅ Pass completion chart saved!")

In [ ]:
# Progressive Passes - Who moves ball forward most?
df_passes['start_x'] = df_passes['location'].apply(
    lambda loc: loc[0] if isinstance(loc, list) else None)
df_passes['end_x'] = df_passes['pass_end_location'].apply(
    lambda loc: loc[0] if isinstance(loc, list) else None)

df_passes['progressive'] = (df_passes['end_x'] - df_passes['start_x']) >= 10

prog_passes = df_passes[df_passes['progressive'] == True].groupby(
    ['player_name','team_name']).size().reset_index(name='Progressive_Passes')
prog_passes = prog_passes[prog_passes['Progressive_Passes'] >= 20].sort_values(
    'Progressive_Passes', ascending=False)

print("✅ Top Progressive Passers:")
print(prog_passes.head(15).to_string(index=False))

In [ ]:
# Progressive Passes Chart
top_prog = prog_passes.head(15).sort_values('Progressive_Passes')

fig, ax = plt.subplots(figsize=(12, 8))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')

colors = ['#ff4757' if team == 'Real Madrid' else '#00d4ff'
          for team in top_prog['team_name']]

ax.barh(top_prog['player_name'], top_prog['Progressive_Passes'],
        color=colors, alpha=0.85)

ax.set_xlabel('Progressive Passes (≥10 yards forward)', color='white', fontsize=12)
ax.set_title('Progressive Passes — Top Players\nLa Liga 2015/16 | Barcelona & Real Madrid',
             color='white', fontsize=14, fontweight='bold')
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_edgecolor('#333333')

blue_patch = mpatches.Patch(color='#00d4ff', alpha=0.85, label='Barcelona')
red_patch = mpatches.Patch(color='#ff4757', alpha=0.85, label='Real Madrid')
ax.legend(handles=[blue_patch, red_patch], facecolor='#1a1a2e', labelcolor='white')

plt.tight_layout()
plt.savefig('progressive_passes.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("✅ Progressive passes chart saved!")

## 🕸️ Pass Networks

In [ ]:
# Pass Network - Barcelona (Top 11 players by pass count)
barca_passes = df_passes[
    (df_passes['team_name'] == 'Barcelona') &
    (df_passes['pass_outcome_name'].isna())
].copy()

# Top 11 players only
top11_barca = barca_passes['player_name'].value_counts().head(11).index.tolist()
barca_passes = barca_passes[barca_passes['player_name'].isin(top11_barca)]

avg_positions = barca_passes.groupby('player_name').agg(
    x=('location', lambda s: np.mean([loc[0] for loc in s if isinstance(loc, list)])),
    y=('location', lambda s: np.mean([loc[1] for loc in s if isinstance(loc, list)])),
    count=('player_name', 'count')
).reset_index()

barca_passes['recipient'] = barca_passes['pass_recipient_name']
pass_pairs = barca_passes.groupby(['player_name','recipient']).size().reset_index(name='passes')
pass_pairs = pass_pairs[
    (pass_pairs['passes'] >= 20) &
    (pass_pairs['recipient'].isin(top11_barca))
]

pitch = Pitch(pitch_type='statsbomb', pitch_color='#0d1117', line_color='#ffffff')
fig, ax = pitch.draw(figsize=(14, 10))
ax.invert_xaxis()
fig.patch.set_facecolor('#0d1117')

for _, row in pass_pairs.iterrows():
    if row['player_name'] in avg_positions['player_name'].values and \
       row['recipient'] in avg_positions['player_name'].values:
        x_start = avg_positions[avg_positions['player_name']==row['player_name']]['x'].values[0]
        y_start = avg_positions[avg_positions['player_name']==row['player_name']]['y'].values[0]
        x_end   = avg_positions[avg_positions['player_name']==row['recipient']]['x'].values[0]
        y_end   = avg_positions[avg_positions['player_name']==row['recipient']]['y'].values[0]
        alpha   = min(row['passes']/150, 0.9)
        lw      = row['passes']/60
        ax.plot([x_start, x_end],[y_start, y_end],
                color='#00d4ff', alpha=alpha, linewidth=lw, zorder=2)

for _, row in avg_positions.iterrows():
    ax.scatter(row['x'], row['y'], s=row['count']/2,
               color='#ff4757', zorder=4, edgecolors='white', linewidth=2)
    short_name = row['player_name'].split()[-1]
    ax.annotate(short_name, (row['x'], row['y']),
                textcoords='offset points', xytext=(0, 12),
                color='white', fontsize=9, ha='center', fontweight='bold')

ax.set_title('Barcelona — Pass Network (Top 11)\nLa Liga 2015/16',
             color='white', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('barca_pass_network.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("✅ Barcelona Pass Network saved!")

In [ ]:
# Pass Network - Real Madrid (Top 11 players by pass count)
rm_passes = df_passes[
    (df_passes['team_name'] == 'Real Madrid') &
    (df_passes['pass_outcome_name'].isna())
].copy()

top11_rm = rm_passes['player_name'].value_counts().head(11).index.tolist()
rm_passes = rm_passes[rm_passes['player_name'].isin(top11_rm)]

avg_positions_rm = rm_passes.groupby('player_name').agg(
    x=('location', lambda s: np.mean([loc[0] for loc in s if isinstance(loc, list)])),
    y=('location', lambda s: np.mean([loc[1] for loc in s if isinstance(loc, list)])),
    count=('player_name', 'count')
).reset_index()

rm_passes['recipient'] = rm_passes['pass_recipient_name']
pass_pairs_rm = rm_passes.groupby(['player_name','recipient']).size().reset_index(name='passes')
pass_pairs_rm = pass_pairs_rm[
    (pass_pairs_rm['passes'] >= 20) &
    (pass_pairs_rm['recipient'].isin(top11_rm))
]

pitch = Pitch(pitch_type='statsbomb', pitch_color='#0d1117', line_color='#ffffff')
fig, ax = pitch.draw(figsize=(14, 10))
ax.invert_xaxis()
fig.patch.set_facecolor('#0d1117')

for _, row in pass_pairs_rm.iterrows():
    if row['player_name'] in avg_positions_rm['player_name'].values and \
       row['recipient'] in avg_positions_rm['player_name'].values:
        x_start = avg_positions_rm[avg_positions_rm['player_name']==row['player_name']]['x'].values[0]
        y_start = avg_positions_rm[avg_positions_rm['player_name']==row['player_name']]['y'].values[0]
        x_end   = avg_positions_rm[avg_positions_rm['player_name']==row['recipient']]['x'].values[0]
        y_end   = avg_positions_rm[avg_positions_rm['player_name']==row['recipient']]['y'].values[0]
        alpha   = min(row['passes']/150, 0.9)
        lw      = row['passes']/60
        ax.plot([x_start, x_end],[y_start, y_end],
                color='#ff4757', alpha=alpha, linewidth=lw, zorder=2)

for _, row in avg_positions_rm.iterrows():
    ax.scatter(row['x'], row['y'], s=row['count']/2,
               color='#00d4ff', zorder=4, edgecolors='white', linewidth=2)
    short_name = row['player_name'].split()[-1]
    ax.annotate(short_name, (row['x'], row['y']),
                textcoords='offset points', xytext=(0, 12),
                color='white', fontsize=9, ha='center', fontweight='bold')

ax.set_title('Real Madrid — Pass Network (Top 11)\nLa Liga 2015/16',
             color='white', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('rm_pass_network.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("✅ Real Madrid Pass Network saved!")

In [ ]:
# Messi vs Ronaldo Head-to-Head Summary
messi = xg_summary[xg_summary['player_name'].str.contains('Messi', na=False)].iloc[0]
cr7   = xg_summary[xg_summary['player_name'].str.contains('Cristiano', na=False)].iloc[0]

categories = ['Total Shots', 'Goals', 'xG', 'xG/Shot', 'Goals vs xG']
messi_vals = [messi['Total_Shots'], messi['Goals'],
              round(messi['Total_xG'],1), round(messi['xG_per_Shot'],3),
              round(messi['Goals_vs_xG'],1)]
cr7_vals   = [cr7['Total_Shots'], cr7['Goals'],
              round(cr7['Total_xG'],1), round(cr7['xG_per_Shot'],3),
              round(cr7['Goals_vs_xG'],1)]

print("="*55)
print(f"{'Metric':<20} {'Messi':>15} {'Ronaldo':>15}")
print("="*55)
for cat, m, c in zip(categories, messi_vals, cr7_vals):
    print(f"{cat:<20} {str(m):>15} {str(c):>15}")
print("="*55)

In [ ]:
# Messi vs Ronaldo Visual Comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.patch.set_facecolor('#0d1117')

metrics    = ['Total_Shots', 'Goals', 'Total_xG']
labels     = ['Total Shots', 'Goals', 'Total xG']
messi_data = [messi['Total_Shots'], messi['Goals'], round(messi['Total_xG'],1)]
cr7_data   = [cr7['Total_Shots'],   cr7['Goals'],   round(cr7['Total_xG'],1)]

for i, (ax, metric, label) in enumerate(zip(axes, metrics, labels)):
    ax.set_facecolor('#0d1117')
    bars = ax.bar(['Messi','Ronaldo'], [messi_data[i], cr7_data[i]],
                  color=['#00d4ff','#ff4757'], alpha=0.85, width=0.5)
    for bar, val in zip(bars, [messi_data[i], cr7_data[i]]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(val), ha='center', va='bottom', color='white',
                fontsize=12, fontweight='bold')
    ax.set_title(label, color='white', fontsize=12, fontweight='bold')
    ax.tick_params(colors='white')
    ax.set_facecolor('#0d1117')
    for spine in ax.spines.values():
        spine.set_edgecolor('#333333')

fig.suptitle('Messi vs Ronaldo — La Liga 2015/16',
             color='white', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('messi_vs_cr7.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("✅ Messi vs Ronaldo comparison saved!")

## 🔥 Activity Heatmaps | Messi vs Ronaldo

In [ ]:
# Messi Activity Heatmap
messi_events = df_master_events[
    df_master_events['player_name'].str.contains('Messi', na=False)
].copy()

messi_events['x'] = messi_events['location'].apply(
    lambda loc: loc[0] if isinstance(loc, list) else None)
messi_events['y'] = messi_events['location'].apply(
    lambda loc: loc[1] if isinstance(loc, list) else None)
messi_events.dropna(subset=['x','y'], inplace=True)

pitch = Pitch(pitch_type='statsbomb', pitch_color='#0d1117', line_color='#ffffff')
fig, ax = pitch.draw(figsize=(14, 10))
fig.patch.set_facecolor('#0d1117')

pitch.kdeplot(messi_events['x'], messi_events['y'], ax=ax,
              cmap='Blues', fill=True, levels=100, alpha=0.7, zorder=2)

ax.set_title('Lionel Messi — Activity Heatmap\nLa Liga 2015/16',
             color='white', fontsize=14, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig('messi_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f"✅ Messi Heatmap saved! Total events: {len(messi_events)}")

In [ ]:
# Ronaldo Activity Heatmap
cr7_events = df_master_events[
    df_master_events['player_name'].str.contains('Cristiano', na=False)
].copy()

cr7_events['x'] = cr7_events['location'].apply(
    lambda loc: loc[0] if isinstance(loc, list) else None)
cr7_events['y'] = cr7_events['location'].apply(
    lambda loc: loc[1] if isinstance(loc, list) else None)
cr7_events.dropna(subset=['x','y'], inplace=True)

pitch = Pitch(pitch_type='statsbomb', pitch_color='#0d1117', line_color='#ffffff')
fig, ax = pitch.draw(figsize=(14, 10))
fig.patch.set_facecolor('#0d1117')

pitch.kdeplot(cr7_events['x'], cr7_events['y'], ax=ax,
              cmap='Reds', fill=True, levels=100, alpha=0.7, zorder=2)

ax.set_title('Cristiano Ronaldo — Activity Heatmap\nLa Liga 2015/16',
             color='white', fontsize=14, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig('cr7_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f"✅ Ronaldo Heatmap saved! Total events: {len(cr7_events)}")

In [ ]:
# Side-by-Side Heatmap Comparison
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.patch.set_facecolor('#0d1117')

pitch = Pitch(pitch_type='statsbomb', pitch_color='#0d1117', line_color='#ffffff')

# Messi
pitch.draw(ax=axes[0])
pitch.kdeplot(messi_events['x'], messi_events['y'], ax=axes[0],
              cmap='Blues', fill=True, levels=100, alpha=0.7, zorder=2)
axes[0].set_title('Lionel Messi', color='white', fontsize=14, fontweight='bold')
axes[0].set_facecolor('#0d1117')

# Ronaldo
pitch.draw(ax=axes[1])
pitch.kdeplot(cr7_events['x'], cr7_events['y'], ax=axes[1],
              cmap='Reds', fill=True, levels=100, alpha=0.7, zorder=2)
axes[1].set_title('Cristiano Ronaldo', color='white', fontsize=14, fontweight='bold')
axes[1].set_facecolor('#0d1117')

fig.suptitle('Activity Heatmap Comparison — La Liga 2015/16',
             color='white', fontsize=16, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('heatmap_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("✅ Side-by-side heatmap comparison saved!")

In [ ]:
# Export data for Power BI
# Shots data
df_shots_export = df_shots[[
    'player_name','team_name','minute',
    'shot_statsbomb_xg','shot_outcome_name',
    'shot_technique_name','shot_body_part_name'
]].copy()
df_shots_export['x'] = df_shots['location'].apply(
    lambda loc: loc[0] if isinstance(loc, list) else None)
df_shots_export['y'] = df_shots['location'].apply(
    lambda loc: loc[1] if isinstance(loc, list) else None)
df_shots_export['is_goal'] = (df_shots_export['shot_outcome_name'] == 'Goal').astype(int)
df_shots_export.to_csv('shots_data.csv', index=False)

# Passes data
df_passes_export = df_passes[[
    'player_name','team_name','minute',
    'pass_length','pass_angle',
    'pass_outcome_name','position_name'
]].copy()
df_passes_export['start_x'] = df_passes['location'].apply(
    lambda loc: loc[0] if isinstance(loc, list) else None)
df_passes_export['start_y'] = df_passes['location'].apply(
    lambda loc: loc[1] if isinstance(loc, list) else None)
df_passes_export['completed'] = df_passes_export['pass_outcome_name'].isna().astype(int)
df_passes_export.to_csv('passes_data.csv', index=False)

# xG Summary
xg_summary.to_csv('xg_summary.csv', index=False)

# Pass summary
pass_summary.to_csv('pass_summary.csv', index=False)

print("✅ All CSV files exported!")
print(f"Shots: {len(df_shots_export)} rows")
print(f"Passes: {len(df_passes_export)} rows")

In [ ]:
import os
import shutil

# --- PROJECT STRUCTURE INITIALIZATION ---
# Defining organized directories for a professional data analytics workflow
def initialize_project_structure():
    """Sets up the assets and data directories for better project management."""
    directories = ['assets', 'data']

    for folder in directories:
        if not os.path.exists(folder):
            os.makedirs(folder)
            print(f"Directory created: {folder}")

    # Sorting generated files into their respective folders
    # Moving PNG visualizations to 'assets/' and CSV datasets to 'data/'
    for file in os.listdir('.'):
        if file.endswith('.png'):
            shutil.move(file, os.path.join('assets', file))
        elif file.endswith('.csv'):
            shutil.move(file, os.path.join('data', file))

    print("Workflow: Files have been successfully organized into assets and data directories.")

# Execute the organization logic
if __name__ == "__main__":
    initialize_project_structure()

# --- VERSION CONTROL NOTE ---
# Project deployment to GitHub is managed via Git CLI.
# Sensitive credentials (Tokens/Emails) are handled through environment variables
# to ensure security and prevent data leakage in public repositories.